<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/airport_agent_masterclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Friday Problem: building an AI airport analyst with Gemini

**Decoding Data Science | Sunday Agentic Masterclass**

All data and documents in this notebook are **fictional**. There is no real airport, no real passenger and no real policy behind them.

## The story

A general international airport is growing fast. The operations manager has one worry: "Friday evenings feel wrong. Queues are long, people complain, and I suspect we are losing money. But nobody can tell me how much, and nobody can tell me which of our own rules we are breaking."

The answer lives in two places that never talk to each other:

- **Numbers** in a database (flights, journeys, staffing, feedback, revenue)
- **Rules** in documents (service standards, complaint procedure, incentive policy, delay SOP)

We will build one agent that reads both, and ask it the manager's question.

## What we build, mapped to the Four-Layer Shift

| Layer | What it means here | Where in this notebook |
|---|---|---|
| Prompt | Role, rules, answer format | Step 6, `SYSTEM_PROMPT` |
| Context | Database schema + policy documents | Steps 2, 4 and 5 |
| Harness | Tools with guardrails (read-only SQL, retrieval) and visible tool calls | Steps 4 and 5 |
| Loop | The agent decides which tool to call, reads results, tries again | Steps 6 and 7 |

## Models used

- Reasoning and tool use: Gemini Flash (the notebook picks the newest available from a short list)
- Embeddings for RAG: `gemini-embedding-001`

## Step 1. Setup

**Say:** "Three libraries and one API key. The key lives in Colab Secrets, never in the notebook."

Before running: in Colab click the key icon on the left, add a secret named `GEMINI_API_KEY`, and switch on notebook access.

In [1]:
!pip install -q google-genai gradio

In [16]:
import os, getpass
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    API_KEY = userdata.get("gemini2")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

# Newest first. The first one that answers is used.
MODEL_CANDIDATES = ["gemini-3.8-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite"]
MODEL = None
for m in MODEL_CANDIDATES:
    try:
        r = client.models.generate_content(model=m, contents="Reply with one word: ready")
        MODEL = m
        print(f"Using {m}: {r.text.strip()}")
        break
    except Exception as e:
        print(f"{m} not available: {str(e)[:150]}")
assert MODEL, "No Gemini model reachable. Check the API key and billing."

EMBED_MODEL = "gemini-embedding-001"

Using gemini-3.8-flash: ready


## Step 2. The data: a fictional airport

**Say:** "We generate the data ourselves. That is the point: you can build this for any department without touching a single real record."

The generator below creates six months (April to September 2026) of flights, sampled passenger journeys, staffing, survey feedback and revenue. A fixed seed means everyone gets identical numbers.

Hidden in the data, on purpose: something is wrong on Friday evenings. We will let the agent find it.

In [3]:
%%writefile airport_data.py
"""Synthetic data generator for a fictional, unnamed 'General International Airport'.

All data is fictional. Fixed seed, so every run produces identical numbers.
Planted story: Friday evenings (17:00-22:00) have only 5 departure immigration counters
open instead of 9, so waits explode, passengers shop less, satisfaction drops, a few miss
flights. Also planted: one chronically late airline, one very high load-factor route
with very low frequency.
"""
import sqlite3

import numpy as np
import pandas as pd

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]


def _lognormal(rng, mean, sigma, size=None):
    mean = np.asarray(mean, dtype=float)
    mu = np.log(mean) - sigma**2 / 2
    return rng.lognormal(mu, sigma, size if size is not None else mean.shape)


def build_airport_db(db_path="airport.db", seed=42, start="2026-04-01", end="2026-09-30"):
    rng = np.random.default_rng(seed)
    days = pd.date_range(start, end, freq="D")

    # ------------------------------------------------------------------ flights
    routes = [
        # destination, weekly flights per direction, base load factor, main nationality, seats
        ("Mumbai", 28, 0.84, "India", 180),
        ("Delhi", 21, 0.82, "India", 180),
        ("Kochi", 14, 0.80, "India", 162),
        ("Karachi", 21, 0.80, "Pakistan", 180),
        ("Cairo", 14, 0.78, "Egypt", 180),
        ("Dhaka", 14, 0.83, "Bangladesh", 189),
        ("Colombo", 4, 0.95, "Sri Lanka", 162),  # planted: full planes, very few flights
        ("Kathmandu", 7, 0.66, "Nepal", 144),
        ("Manila", 7, 0.77, "Philippines", 180),
        ("Amman", 7, 0.72, "Jordan", 162),
        ("Istanbul", 7, 0.74, "Turkey", 189),
        ("Nairobi", 5, 0.70, "Kenya", 162),
        ("London", 7, 0.81, "United Kingdom", 300),
    ]
    airlines = ["Skyline Air", "Desert Falcon", "AeroNova", "Meridian Air", "Crescent Airways"]
    airline_p = [0.25, 0.20, 0.20, 0.20, 0.15]
    airline_delay_p = {"Skyline Air": 0.20, "Desert Falcon": 0.22, "AeroNova": 0.25,
                       "Meridian Air": 0.18, "Crescent Airways": 0.55}  # planted: Crescent is chronic

    dep_w = np.array([3, 3, 2, 2, 3, 5, 8, 9, 8, 6, 5, 5, 5, 5, 5, 6, 7, 9, 10, 10, 9, 8, 6, 4], float)
    arr_w = np.array([5, 5, 5, 5, 6, 8, 9, 8, 6, 5, 5, 5, 6, 7, 8, 8, 7, 6, 6, 7, 8, 8, 7, 6], float)

    rows = []
    for d in days:
        wd = d.weekday()
        for dest, weekly, base_lf, nat, seats in routes:
            for direction in ("Arrival", "Departure"):
                n = rng.poisson(weekly / 7)
                for _ in range(n):
                    w = (dep_w if direction == "Departure" else arr_w).copy()
                    if direction == "Departure" and wd == 4:  # Friday evening rush
                        w[17:23] *= 1.6
                    hour = int(rng.choice(24, p=w / w.sum()))
                    minute = int(rng.choice(range(0, 60, 5)))
                    rows.append((d, wd, hour, minute, dest, direction, seats, base_lf))
    f = pd.DataFrame(rows, columns=["flight_date", "wd", "scheduled_hour", "minute", "route",
                                    "direction", "seats", "base_lf"])
    f = f.sort_values(["flight_date", "scheduled_hour", "minute"]).reset_index(drop=True)
    n = len(f)
    f["flight_id"] = ["F%06d" % (i + 1) for i in range(n)]
    f["weekday"] = f["wd"].map(lambda i: WEEKDAYS[i])
    f["scheduled_time"] = f["scheduled_hour"].map("{:02d}".format) + ":" + f["minute"].map("{:02d}".format)
    f["airline"] = rng.choice(airlines, size=n, p=airline_p)

    month = f["flight_date"].dt.month
    lf = f["base_lf"] + rng.normal(0, 0.04, n) + np.where(month.isin([7, 8]), 0.04, 0.0)
    lf = lf + np.where(f["wd"].isin([4, 6]), 0.03, 0.0)
    lf = np.clip(lf, 0.40, 0.99)
    f["passengers"] = np.round(f["seats"] * lf).astype(int)

    p_delay = f["airline"].map(airline_delay_p).to_numpy()
    delayed = rng.random(n) < p_delay
    delay = np.where(delayed, rng.exponential(45, n) + 16, np.maximum(0, rng.normal(3, 4, n)))
    f["delay_min"] = np.round(delay).astype(int)

    reasons = ["Late inbound aircraft", "Technical", "Crew", "Ground handling", "Weather"]
    w_crescent = [0.40, 0.35, 0.15, 0.05, 0.05]
    w_other = [0.30, 0.15, 0.15, 0.20, 0.20]
    reason = np.full(n, "", dtype=object)
    is_cres = (f["airline"] == "Crescent Airways").to_numpy()
    for mask, wts in ((is_cres, w_crescent), (~is_cres, w_other)):
        idx = np.where(mask & (f["delay_min"].to_numpy() > 15))[0]
        reason[idx] = rng.choice(reasons, size=len(idx), p=wts)
    f["delay_reason"] = np.where(reason == "", None, reason)

    cancelled = rng.random(n) < 0.012
    f.loc[cancelled, ["delay_min", "passengers"]] = 0
    f.loc[cancelled, "delay_reason"] = None
    f["status"] = np.where(cancelled, "Cancelled", np.where(f["delay_min"] > 15, "Delayed", "On time"))
    f["load_factor"] = np.where(f["passengers"] > 0, (f["passengers"] / f["seats"]).round(3), 0.0)
    f["flight_date"] = f["flight_date"].dt.strftime("%Y-%m-%d")

    flights = f[["flight_id", "flight_date", "weekday", "scheduled_time", "scheduled_hour", "airline",
                 "route", "direction", "seats", "passengers", "load_factor", "delay_min",
                 "delay_reason", "status"]].copy()

    # ----------------------------------------------------------------- staffing
    srows = []
    for d in days:
        wd = d.weekday()
        for h in range(24):
            if h <= 4:
                dep_c, dep_m = 4, 0.30
            elif h <= 6:
                dep_c, dep_m = 7, 0.80
            elif h <= 16:
                dep_c, dep_m = 8, 1.00
            elif h <= 22:
                dep_c, dep_m = 9, 1.00
            else:
                dep_c, dep_m = 6, 0.60
            if wd == 4 and 17 <= h <= 22:  # planted: Friday-evening roster gap
                dep_c, dep_m = 5, 1.45
            elif wd == 6 and 17 <= h <= 21:
                dep_m = 1.20
            arr_c = 3 if h <= 4 else 6
            srows.append((d.strftime("%Y-%m-%d"), h, "Departure", dep_c, dep_m))
            srows.append((d.strftime("%Y-%m-%d"), h, "Arrival", arr_c, 1.0))
    staffing_full = pd.DataFrame(srows, columns=["flight_date", "hour", "direction",
                                                 "counters_open", "demand_mult"])
    staffing = staffing_full[["flight_date", "hour", "direction", "counters_open"]].copy()

    # ------------------------------------------------------- passenger journeys
    ok = flights[flights["passengers"] > 0].reset_index(drop=True)
    ns = rng.binomial(ok["passengers"].to_numpy(), 0.03)
    j = ok.loc[np.repeat(ok.index.to_numpy(), ns),
               ["flight_id", "flight_date", "weekday", "scheduled_hour", "direction", "route"]].reset_index(drop=True)
    m = len(j)
    j.insert(0, "journey_id", ["J%07d" % (i + 1) for i in range(m)])

    main_nat = {r[0]: r[3] for r in routes}
    glob_nat = ["India", "Pakistan", "Egypt", "Philippines", "Bangladesh", "Sri Lanka", "Nepal",
                "Jordan", "United Kingdom", "Kenya", "Turkey", "Other"]
    glob_p = np.array([14, 10, 7, 8, 8, 4, 3, 3, 8, 3, 3, 29], float)
    glob_p /= glob_p.sum()
    is_main = rng.random(m) < 0.55
    nat = rng.choice(glob_nat, size=m, p=glob_p)
    j["nationality"] = np.where(is_main, j["route"].map(main_nat), nat)

    visas = ["Tourist", "Visit", "Residence", "Work Permit", "Transit"]
    p_arr = [0.28, 0.22, 0.30, 0.15, 0.05]
    p_dep = [0.20, 0.17, 0.38, 0.20, 0.05]
    visa = np.empty(m, dtype=object)
    arr_mask = (j["direction"] == "Arrival").to_numpy()
    visa[arr_mask] = rng.choice(visas, size=int(arr_mask.sum()), p=p_arr)
    visa[~arr_mask] = rng.choice(visas, size=int((~arr_mask).sum()), p=p_dep)
    j["visa_type"] = visa

    j["hour"] = j["scheduled_hour"]
    j = j.merge(staffing_full[staffing_full["direction"] == "Departure"]
                .rename(columns={"hour": "hour"}), on=["flight_date", "hour", "direction"], how="left")
    dep_mask = (j["direction"] == "Departure").to_numpy()
    counters = j["counters_open"].fillna(6).to_numpy()
    dm = j["demand_mult"].fillna(1.0).to_numpy()
    imm_mean = np.where(dep_mask, 11.0 * (8.0 / counters) ** 1.8 * dm, 9.0)
    imm = np.round(_lognormal(rng, imm_mean, 0.5)).clip(1, 150)
    chk = np.where(dep_mask, np.round(_lognormal(rng, np.full(m, 9.0), 0.5)).clip(1, 90), np.nan)
    sec = np.where(dep_mask, np.round(_lognormal(rng, np.full(m, 7.0), 0.5)).clip(1, 60), np.nan)
    bag = np.where(~dep_mask, np.round(_lognormal(rng, np.full(m, 15.0), 0.4)).clip(3, 90), np.nan)
    j["check_in_wait_min"] = chk
    j["security_wait_min"] = sec
    j["immigration_wait_min"] = imm
    j["baggage_wait_min"] = bag
    j["total_processing_min"] = np.where(dep_mask, chk + sec + imm, imm + bag)

    base_spend = j["visa_type"].map({"Tourist": 85, "Visit": 70, "Residence": 55,
                                     "Work Permit": 35, "Transit": 95}).to_numpy(dtype=float)
    dwell = np.clip((120 - j["total_processing_min"].to_numpy()) / 90, 0.25, 1.15)
    buys = rng.random(m) < np.where(dep_mask, 0.78, 0.5)
    spend_dep = base_spend * _lognormal(rng, np.ones(m), 0.5) * dwell
    spend_arr = 12.0 * _lognormal(rng, np.ones(m), 0.5)
    j["spend_aed"] = np.round(np.where(buys, np.where(dep_mask, spend_dep, spend_arr), 0.0), 1)
    p_miss = np.where(dep_mask & (imm > 45), 0.10, np.where(dep_mask, 0.004, 0.0))
    j["missed_flight"] = (rng.random(m) < p_miss).astype(int)

    journeys = j[["journey_id", "flight_id", "flight_date", "weekday", "hour", "direction", "nationality",
                  "visa_type", "check_in_wait_min", "security_wait_min", "immigration_wait_min",
                  "baggage_wait_min", "total_processing_min", "spend_aed", "missed_flight"]].copy()

    # ----------------------------------------------------------------- feedback
    fb_idx = np.where(rng.random(m) < 0.12)[0]
    fbj = j.iloc[fb_idx].reset_index(drop=True)
    k = len(fbj)
    cats = ["Waiting time", "Staff", "Cleanliness", "Shops & dining", "Wi-Fi", "Signage"]
    cat = rng.choice(cats, size=k, p=[0.30, 0.18, 0.12, 0.15, 0.13, 0.12])
    excess = np.maximum(0, fbj["immigration_wait_min"].to_numpy() - 10)
    score_f = np.select(
        [cat == "Waiting time", cat == "Staff", cat == "Cleanliness", cat == "Shops & dining",
         cat == "Wi-Fi", cat == "Signage"],
        [4.6 - 0.07 * excess, 4.3 - 0.03 * excess, np.full(k, 4.4), 3.9 - 0.015 * excess,
         np.full(k, 3.4), np.full(k, 4.0)]) + rng.normal(0, 0.6, k)
    score = np.clip(np.round(score_f), 1, 5).astype(int)

    templates = {
        "Waiting time": {
            "low": ["Waited far too long at immigration, I almost missed my flight.",
                    "The passport queue was endless and only a few counters were open.",
                    "Departures immigration queue was unacceptable.",
                    "الانتظار في الجوازات طويل جدا ولم يبق وقت للتسوق."],
            "mid": ["Wait at passport control was longer than expected.",
                    "Queues were acceptable but could be faster."],
            "high": ["Quick and smooth immigration.", "Very short queue, great experience.",
                     "إجراءات سريعة وسهلة."]},
        "Staff": {
            "low": ["Staff were unhelpful and seemed rushed.", "Nobody could answer my question."],
            "mid": ["Staff were fine, nothing special."],
            "high": ["Very helpful and polite staff.", "الموظفون متعاونون ومحترمون."]},
        "Cleanliness": {
            "low": ["Restrooms were not clean."],
            "mid": ["Terminal cleanliness is average."],
            "high": ["The terminal was spotless.", "المطار نظيف جدا."]},
        "Shops & dining": {
            "low": ["No time to shop and the prices are high.", "Limited food choices near my gate."],
            "mid": ["Average selection of shops."],
            "high": ["Good variety of food and shops."]},
        "Wi-Fi": {
            "low": ["Wi-Fi kept disconnecting.", "الواي فاي ضعيف جدا."],
            "mid": ["Wi-Fi was slow at times."],
            "high": ["Wi-Fi worked well."]},
        "Signage": {
            "low": ["Hard to find my gate, the signs are confusing."],
            "mid": ["Signs are okay but could be clearer."],
            "high": ["Clear signs everywhere."]},
    }
    comments, lang = [], []
    for c, s in zip(cat, score):
        band = "low" if s <= 2 else ("mid" if s == 3 else "high")
        text = str(rng.choice(templates[c][band]))
        comments.append(text)
        lang.append("ar" if any("\u0600" <= ch <= "\u06ff" for ch in text) else "en")
    feedback = pd.DataFrame({
        "feedback_id": ["FB%06d" % (i + 1) for i in range(k)],
        "journey_id": fbj["journey_id"], "flight_date": fbj["flight_date"],
        "weekday": fbj["weekday"], "hour": fbj["hour"], "direction": fbj["direction"],
        "category": cat, "score": score, "language": lang, "comment": comments})

    # ------------------------------------------------------------------ revenue
    day_pax = flights.groupby("flight_date")["passengers"].sum()
    day_arr = flights[flights["direction"] == "Arrival"].groupby("flight_date")["passengers"].sum()
    samp = journeys.groupby("flight_date").agg(n=("journey_id", "count"), spend=("spend_aed", "sum"))
    frac = (samp["n"] / day_pax).reindex(day_pax.index)
    est_spend = (samp["spend"] / frac).reindex(day_pax.index)
    noise = lambda s: rng.normal(1.0, 0.03, len(day_pax))
    rev = pd.DataFrame({
        "flight_date": day_pax.index,
        "passengers": day_pax.values,
        "aeronautical_aed": np.round(day_pax.values * 38 * noise(0)),
        "duty_free_aed": np.round(est_spend.values * 0.55),
        "food_beverage_aed": np.round(est_spend.values * 0.35),
        "other_retail_aed": np.round(est_spend.values * 0.10),
        "parking_aed": np.round(day_pax.values * 1.2 * noise(0)),
        "car_rental_aed": np.round(day_arr.reindex(day_pax.index).values * 2.4 * noise(0)),
        "advertising_aed": np.round(18000 * noise(0)),
    })
    rev["total_revenue_aed"] = rev[["aeronautical_aed", "duty_free_aed", "food_beverage_aed",
                                    "other_retail_aed", "parking_aed", "car_rental_aed",
                                    "advertising_aed"]].sum(axis=1)

    # -------------------------------------------------------------------- save
    con = sqlite3.connect(db_path)
    for name, df in (("flights", flights), ("passenger_journeys", journeys), ("staffing", staffing),
                     ("feedback", feedback), ("daily_revenue", rev)):
        df.to_sql(name, con, if_exists="replace", index=False)
    con.execute("CREATE INDEX IF NOT EXISTS ix_f_date ON flights(flight_date)")
    con.execute("CREATE INDEX IF NOT EXISTS ix_j_date ON passenger_journeys(flight_date, hour)")
    con.commit()
    con.close()
    return {"flights": len(flights), "journeys": len(journeys), "staffing": len(staffing),
            "feedback": len(feedback), "daily_revenue": len(rev)}


if __name__ == "__main__":
    print(build_airport_db("/tmp/work/airport.db"))

Overwriting airport_data.py


In [17]:
import sqlite3
import pandas as pd
from airport_data import build_airport_db

DB_PATH = "airport.db"
print(build_airport_db(DB_PATH))

con = sqlite3.connect(DB_PATH)
for table in ["flights", "passenger_journeys", "staffing", "feedback", "daily_revenue"]:
    print("\n==", table)
    display(pd.read_sql(f"SELECT * FROM {table} LIMIT 3", con))
con.close()

{'flights': 8177, 'journeys': 36155, 'staffing': 8784, 'feedback': 4242, 'daily_revenue': 183}

== flights


,flight_id,flight_date,weekday,scheduled_time,scheduled_hour,airline,route,direction,seats,passengers,load_factor,delay_min,delay_reason,status
0,F000001,2026-04-01,Wednesday,03:35,3,Crescent Airways,Karachi,Arrival,180,146,0.811,101,Technical,Delayed
1,F000002,2026-04-01,Wednesday,04:00,4,Meridian Air,Cairo,Departure,180,140,0.778,6,None,On time
2,F000003,2026-04-01,Wednesday,04:25,4,AeroNova,Dhaka,Arrival,189,164,0.868,4,None,On time



== passenger_journeys


,journey_id,flight_id,flight_date,weekday,hour,direction,nationality,visa_type,check_in_wait_min,security_wait_min,immigration_wait_min,baggage_wait_min,total_processing_min,spend_aed,missed_flight
0,J0000001,F000001,2026-04-01,Wednesday,3,Arrival,Pakistan,Work Permit,NaN,NaN,13.0,19.0,32.0,0.0,0
1,J0000002,F000001,2026-04-01,Wednesday,3,Arrival,Other,Work Permit,NaN,NaN,17.0,15.0,32.0,22.5,0
2,J0000003,F000002,2026-04-01,Wednesday,4,Departure,United Kingdom,Transit,5.0,6.0,16.0,NaN,27.0,71.7,0



== staffing


,flight_date,hour,direction,counters_open
0,2026-04-01,0,Departure,4
1,2026-04-01,0,Arrival,3
2,2026-04-01,1,Departure,4



== feedback


,feedback_id,journey_id,flight_date,weekday,hour,direction,category,score,language,comment
0,FB000001,J0000010,2026-04-01,Wednesday,4,Arrival,Wi-Fi,3,en,Wi-Fi was slow at times.
1,FB000002,J0000012,2026-04-01,Wednesday,4,Arrival,Waiting time,5,en,Quick and smooth immigration.
2,FB000003,J0000029,2026-04-01,Wednesday,7,Departure,Staff,5,ar,الموظفون متعاونون ومحترمون.



== daily_revenue


,flight_date,passengers,aeronautical_aed,duty_free_aed,food_beverage_aed,other_retail_aed,parking_aed,car_rental_aed,advertising_aed,total_revenue_aed
0,2026-04-01,6330,240185.0,109773.0,69856.0,19959.0,7421.0,6677.0,18737.0,472608.0
1,2026-04-02,6765,249493.0,98420.0,62631.0,17895.0,8263.0,7102.0,18027.0,461831.0
2,2026-04-03,6723,254058.0,95427.0,60726.0,17350.0,8201.0,9463.0,16748.0,461973.0


## Step 3. Why an LLM alone is not enough

**Say:** "Watch how confident this sounds. Now remember: it has never seen our airport."

This is the "confident wrong answer" moment. The model has no access to our data, so any number it gives is invented.

In [18]:
question = "What was the average departure immigration wait at our airport on Friday evenings last quarter?"
response = client.models.generate_content(model=MODEL, contents=question)
print(response.text)

I don’t have access to your organization's internal operational data or systems, and I don't know which airport you are referring to. 

To help point you in the right direction:

1. **If you are an airport employee or stakeholder:** This data is typically tracked via queue-management systems (such as LiDAR, stereo-vision cameras, Wi-Fi/Bluetooth beacon tracking, or boarding pass scan-to-scan delta times) and is usually accessible through your airport’s Business Intelligence (BI) dashboard, operations control center (AOCC), or directly from the local immigration/border control agency.
2. **If this data is publicly reported:** Please let me know **which airport** and **which specific quarter/year** you are asking about, and I can check if that authority publishes historical wait-time statistics.


## Step 4. Tool 1: let the agent query the database (Context + Harness)

**Say:** "We give the model two things: a map of the database, and a tool that can run a query. But the tool is read-only and limited, because an agent with a database is a risk until you put guardrails around it."

Guardrails in the code below:
- only a single `SELECT` (or `WITH`) statement is accepted
- write keywords are blocked
- the database is opened in read-only mode
- results are capped at 40 rows
- every call is logged so we can show what the agent did

In [6]:
SCHEMA = """
Database: airport.db (SQLite). Fictional general international airport, 1 April 2026 to 30 September 2026.

TABLE flights  (one row per scheduled flight)
  flight_id, flight_date (YYYY-MM-DD), weekday (Monday..Sunday), scheduled_time (HH:MM), scheduled_hour (0-23),
  airline, route (destination city), direction ('Arrival' or 'Departure'), seats, passengers, load_factor (0-1),
  delay_min, delay_reason, status ('On time', 'Delayed', 'Cancelled')

TABLE passenger_journeys  (a random ~3% SAMPLE of passengers: multiply counts and sums by about 33 to estimate totals)
  journey_id, flight_id, flight_date, weekday, hour (0-23), direction ('Arrival' or 'Departure'), nationality,
  visa_type ('Tourist', 'Visit', 'Residence', 'Work Permit', 'Transit'),
  check_in_wait_min and security_wait_min (departures only),
  immigration_wait_min (departure immigration for departures, arrival immigration for arrivals),
  baggage_wait_min (arrivals only), total_processing_min,
  spend_aed (shopping and dining spend in AED), missed_flight (1 if the passenger missed the flight)

TABLE staffing  (one row per date, hour and direction)
  flight_date, hour (0-23), direction, counters_open (immigration counters open)

TABLE feedback  (survey responses, about 12% of sampled journeys)
  feedback_id, journey_id, flight_date, weekday, hour, direction,
  category ('Waiting time', 'Staff', 'Cleanliness', 'Shops & dining', 'Wi-Fi', 'Signage'),
  score (1-5), language ('en' or 'ar'), comment

TABLE daily_revenue  (one row per date, all amounts in AED)
  flight_date, passengers, aeronautical_aed, duty_free_aed, food_beverage_aed, other_retail_aed,
  parking_aed, car_rental_aed, advertising_aed, total_revenue_aed
"""

import re

TOOL_LOG = []
FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|alter|create|attach|detach|pragma|replace|vacuum)\b", re.I)


def query_airport_database(sql: str) -> str:
    """Run one read-only SQLite SELECT query on the airport database and return the rows as text.

    Args:
        sql: A single SQLite SELECT statement that uses the table and column names from the schema.
    """
    statement = sql.strip().rstrip(";")
    if not re.match(r"^(select|with)\b", statement, re.I) or FORBIDDEN.search(statement) or ";" in statement:
        result = "ERROR: only a single read-only SELECT statement is allowed."
    else:
        try:
            ro = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
            df = pd.read_sql_query(statement, ro)
            ro.close()
            if len(df) == 0:
                result = "No rows returned."
            else:
                result = df.head(40).round(2).to_string(index=False)
                if len(df) > 40:
                    result += f"\n(showing first 40 of {len(df)} rows)"
        except Exception as e:
            result = f"SQL ERROR: {e}"
    TOOL_LOG.append(("query_airport_database", sql, result))
    return result


# Test the tool by hand before the agent uses it
print(query_airport_database(
    "SELECT weekday, ROUND(AVG(immigration_wait_min), 1) AS avg_wait_min "
    "FROM passenger_journeys WHERE direction = 'Departure' GROUP BY weekday ORDER BY avg_wait_min DESC"
))
print()
print(query_airport_database("DROP TABLE flights"))
TOOL_LOG.clear()

  weekday  avg_wait_min
   Friday          23.5
   Sunday          11.0
  Tuesday          10.3
 Saturday          10.3
   Monday          10.3
Wednesday          10.2
 Thursday          10.2

ERROR: only a single read-only SELECT statement is allowed.


## Step 5. Tool 2: let the agent read the rules (RAG)

**Say:** "Numbers tell us what happened. They cannot tell us whether it was allowed. For that the agent needs to read our policies."

How it works, in four moves:
1. Split each document into sections
2. Turn every section into a vector with Gemini embeddings
3. Turn the question into a vector the same way
4. Return the closest sections, with document and section number so the answer can be checked

In [7]:
import os, numpy as np

os.makedirs("docs", exist_ok=True)

DOCS = {}

DOCS["airline_incentive_policy.txt"] = """
AIRLINE INCENTIVE AND ROUTE DEVELOPMENT POLICY
General International Airport (fictional) - Document AIP-03, Version 3.1
This is a fictional document created for training purposes.

Section 1. Objective
1.1 The airport grows passenger numbers by supporting airlines that add destinations or add frequency on routes where demand already exceeds capacity.

Section 2. Priority growth routes
2.1 A route is a priority growth route when its average load factor is 90 percent or higher over a rolling 3 months and it has fewer than 7 weekly departures.
2.2 The Commercial team publishes the list of priority growth routes every quarter.

Section 3. Incentives
3.1 Added frequency on a priority growth route: 50 percent discount on landing charges for the added flights, for 12 months.
3.2 A new destination not served before: 75 percent discount on landing charges in year one, 50 percent in year two and 25 percent in year three.
3.3 Marketing support of up to AED 150,000 is available for a new destination, matched by the airline.

Section 4. Conditions
4.1 To receive or keep any incentive the airline must have no more than 30 percent of its flights delayed in a month, where delayed means more than 15 minutes late.
4.2 An airline that exceeds this limit loses incentives for the following month, and for 3 consecutive months of breach the incentive agreement is suspended.
4.3 Incentives are reviewed annually by the Commercial Committee.
"""

DOCS["complaint_handling_procedure.txt"] = """
COMPLAINT HANDLING PROCEDURE
General International Airport (fictional) - Document CHP-02, Version 1.4
This is a fictional document created for training purposes.

Section 1. Principles
1.1 Every complaint and every survey comment is treated as operational data, not only as a customer issue.
1.2 Comments in any language, including Arabic and English, are handled with the same priority.

Section 2. Handling steps
2.1 Acknowledge every complaint within 1 working day.
2.2 Classify each complaint by theme: waiting time, staff, cleanliness, shops and dining, Wi-Fi, or signage.
2.3 Resolve individual complaints within 7 working days and record the outcome.

Section 3. Systemic issues
3.1 When the same theme appears in 10 or more complaints within 30 days, or when it appears repeatedly in the same time slot, it is a systemic issue.
3.2 A systemic issue requires a root cause report to the Customer Experience Committee within 14 days.
3.3 The root cause report must include the affected time slots, the number of passengers affected, the estimated commercial impact, the proposed correction and the owner of the correction.

Section 4. Support for affected passengers
4.1 A passenger who missed a flight because of an airport queue longer than 45 minutes is eligible for rebooking assistance. The airport coordinates with the airline, and cash compensation is not offered under this procedure.
4.2 Passengers in a delay longer than 120 minutes are eligible for meal vouchers, as set out in the Delay Management SOP.
"""

DOCS["delay_management_sop.txt"] = """
DELAY MANAGEMENT STANDARD OPERATING PROCEDURE
General International Airport (fictional) - Document DMS-04, Version 2.2
This is a fictional document created for training purposes.

Section 1. Definitions
1.1 A flight is delayed when it departs or arrives more than 15 minutes after its scheduled time.
1.2 Every delay longer than 15 minutes must be assigned one reason code: late inbound aircraft, technical, crew, ground handling or weather.

Section 2. Airline performance monitoring
2.1 Operations publishes a punctuality report for every airline each month.
2.2 An airline with more than 30 percent of flights delayed in a month receives a formal warning from the Airline Relations team.
2.3 An airline above the 30 percent limit for 3 consecutive months is placed under performance review, its incentives are suspended under the Airline Incentive Policy, and its slot allocation may be reviewed.
2.4 Delays coded as ground handling that are caused by the airport count against the airport's own performance indicators, not the airline's.

Section 3. Passenger care during delays
3.1 Delay longer than 60 minutes: the airline must provide clear information at the gate and on the information screens.
3.2 Delay longer than 120 minutes: meal vouchers must be offered to every affected passenger.
3.3 Delay longer than 240 minutes: hotel accommodation must be offered to passengers who cannot continue their journey the same day.

Section 4. Root cause follow-up
4.1 When one airline accounts for the majority of technical or late-inbound delays over 2 consecutive months, Operations must meet the airline and agree a recovery plan within 14 days.
"""

DOCS["passenger_service_standards.txt"] = """
PASSENGER SERVICE STANDARDS
General International Airport (fictional) - Document PSS-01, Version 2.0
This is a fictional document created for training purposes.

Section 1. Purpose and scope
1.1 This document sets the minimum service levels that every passenger should experience at the airport, from the moment they arrive at the terminal to the moment they board or leave.
1.2 It applies to the airport operator, border agencies working in the terminal, security contractors and ground handlers.

Section 2. Processing time targets
2.1 Departure check-in: the average wait must not exceed 10 minutes.
2.2 Departure security screening: the average wait must not exceed 10 minutes.
2.3 Departure immigration: the average wait must not exceed 15 minutes, and no more than 5 percent of departing passengers may wait longer than 30 minutes in any peak hour.
2.4 Arrival immigration: the average wait must not exceed 15 minutes.
2.5 Baggage reclaim: the first bag must be on the belt within 20 minutes of the aircraft arriving on stand.

Section 3. Staffing of immigration counters
3.1 Peak departure hours are defined as 17:00 to 22:59 every day of the week.
3.2 During peak departure hours a minimum of 9 departure immigration counters must be open.
3.3 If the average departure immigration wait exceeds 30 minutes for 3 consecutive hours, the airport duty manager must request additional border staff immediately and record the incident.
3.4 Counter rosters must be reviewed every month against the actual waiting times recorded, and any peak hour that failed the target in section 2.3 must be corrected in the next roster.

Section 4. Passenger satisfaction targets
4.1 The average satisfaction score must be at least 4.0 out of 5 in every survey category.
4.2 Any category whose monthly average falls below 3.5 requires a written improvement plan within 14 days.

Section 5. Missed flights
5.1 If the share of departing passengers who miss their flight exceeds 1 percent in any peak hour band, an executive review of staffing and processes is mandatory.
5.2 Airport-caused misses, where the passenger can show that a queue longer than 45 minutes was the cause, must be reported to the Customer Experience Committee.
"""

for name, text in DOCS.items():
    with open(f"docs/{name}", "w", encoding="utf-8") as f:
        f.write(text.strip() + "\n")
print("Documents written:", sorted(os.listdir("docs")))

Documents written: ['airline_incentive_policy.txt', 'complaint_handling_procedure.txt', 'delay_management_sop.txt', 'passenger_service_standards.txt']


In [8]:
def load_chunks(folder="docs"):
    chunks = []
    for fname in sorted(os.listdir(folder)):
        with open(os.path.join(folder, fname), encoding="utf-8") as f:
            text = f.read()
        title = text.split("\n")[0].strip()
        parts = re.split(r"\n(?=Section \d+\.)", text)
        for part in parts[1:]:  # parts[0] is the document header
            part = part.strip()
            chunks.append({"source": title, "section": part.split("\n")[0].strip(), "text": part})
    return chunks


def embed(texts, task_type):
    vectors = []
    for i in range(0, len(texts), 50):
        res = client.models.embed_content(
            model=EMBED_MODEL,
            contents=texts[i:i + 50],
            config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=768),
        )
        vectors += [e.values for e in res.embeddings]
    v = np.array(vectors, dtype="float32")
    return v / np.linalg.norm(v, axis=1, keepdims=True)


CHUNKS = load_chunks()
CHUNK_VECS = embed([f"{c['source']} | {c['text']}" for c in CHUNKS], "RETRIEVAL_DOCUMENT")
print(len(CHUNKS), "sections embedded, vector shape", CHUNK_VECS.shape)


def search_policy_documents(question: str) -> str:
    """Search the airport's policy documents (passenger service standards, complaint handling procedure,
    airline incentive policy, delay management SOP) and return the 3 most relevant sections.

    Args:
        question: What you want to know about a rule, target, procedure or policy, in plain English.
    """
    qv = embed([question], "RETRIEVAL_QUERY")[0]
    scores = CHUNK_VECS @ qv
    top = np.argsort(-scores)[:3]
    result = "\n\n".join(
        f"[{CHUNKS[i]['source']} | {CHUNKS[i]['section']}] (relevance {scores[i]:.2f})\n{CHUNKS[i]['text']}"
        for i in top
    )
    TOOL_LOG.append(("search_policy_documents", question, result))
    return result


# Test retrieval by hand
print(search_policy_documents("How many immigration counters must be open at peak hours?")[:700])
TOOL_LOG.clear()

17 sections embedded, vector shape (17, 768)
[PASSENGER SERVICE STANDARDS | Section 3. Staffing of immigration counters] (relevance 0.80)
Section 3. Staffing of immigration counters
3.1 Peak departure hours are defined as 17:00 to 22:59 every day of the week.
3.2 During peak departure hours a minimum of 9 departure immigration counters must be open.
3.3 If the average departure immigration wait exceeds 30 minutes for 3 consecutive hours, the airport duty manager must request additional border staff immediately and record the incident.
3.4 Counter rosters must be reviewed every month against the actual waiting times recorded, and any peak hour that failed the target in section 2.3 must be corrected in the next roster.

[PASSENGER SERVIC


## Step 6. The agent: prompt + tools + loop

**Say:** "Now we put it together. The system prompt is the job description. The tools are its hands. And the loop, where the model decides which tool to call, reads the result and decides what to do next, is handled for us by the SDK."

Look at rule 1 and rule 2 in the prompt: they stop the model from guessing numbers or inventing policies.

In [20]:
SYSTEM_PROMPT = f"""You are the Airport Operations and Growth Analyst for a fictional general international airport.
You help managers understand operations, passenger experience, revenue and growth.

How you work:
1. For every number, count or comparison, call query_airport_database. Never guess a figure from memory.
2. For every rule, target, procedure or policy, call search_policy_documents and cite the document and section.
3. passenger_journeys is a ~3% sample. When you report totals or revenue impact, scale up by about 33 and say so.
4. When a question needs both facts and rules, use both tools, then compare the actual figures with the rule.
5. If a query fails, read the error, fix the SQL and try again. If the data cannot answer, say so plainly.
6. Answer for a busy manager: finding first, then the evidence (key numbers), then recommended actions.
   Keep it under 250 words unless asked for more.

{SCHEMA}"""

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    tools=[query_airport_database, search_policy_documents],
)
chat = client.chats.create(model=MODEL, config=config)


def ask(question, new_chat=False):
    global chat
    if new_chat:
        chat = client.chats.create(model=MODEL, config=config)
    TOOL_LOG.clear()
    response = chat.send_message(question)
    print("TOOLS CALLED:")
    for name, arg, _ in TOOL_LOG:
        print(f"  - {name}: {arg[:170]}".replace("\n", " "))
    print("\nANSWER:\n" + response.text)

## Step 7. Ask the manager's question, one beat at a time

Each question below is one beat of the story. Run them in order. After each one, point at the **TOOLS CALLED** list: that is the agent working.

### Beat 1: Is the feeling real?

In [21]:
ask("Which weekday has the longest average departure immigration wait, and how does it compare with the other days?")

TOOLS CALLED:
  - query_airport_database: SELECT weekday, ROUND(AVG(immigration_wait_min), 2) AS avg_departure_immigration_wait, COUNT(*) as sample_count FROM passenger_journeys WHERE direction = 'Departure' GROU
  - search_policy_documents: What is the maximum allowed or target waiting time for departure immigration?
  - query_airport_database: SELECT weekday, ROUND(AVG(counters_open), 2) as avg_counters FROM staffing WHERE direction = 'Departure' GROUP BY weekday;
  - query_airport_database: SELECT    CASE strftime('%w', flight_date)     WHEN '0' THEN 'Sunday'     WHEN '1' THEN 'Monday'     WHEN '2' THEN 'Tuesday'     WHEN '3' THEN 'Wednesday'     WHEN '4' TH

ANSWER:
**Finding**
**Friday** has by far the longest average departure immigration wait at **23.46 minutes**—more than double the wait of any other day of the week—and is the only day violating the airport's service standard.

---

### **Evidence**
* **Friday:** 23.46 minutes (sample: 2,706 passenger journeys; represents ~89,3

**Expected:** Friday is the outlier (roughly 23 minutes on average, against about 10 on other days). Feeling confirmed, now zoom in.

### Beat 2: When exactly, and why?

In [ ]:
ask("Look only at Fridays. For each hour from 14:00 to 23:00, show the average departure immigration wait and the average number of counters open, and compare with the same hours on all other days.")

**Expected:** the problem sits in the 17:00 to 22:59 window. On Fridays only 5 counters are open against 9 on other days, and the wait jumps from roughly 10 minutes to over 35.

### Beat 3: What does it cost?

In [ ]:
ask("How much shopping and dining spend per departing passenger do we lose on Friday evenings (17:00 to 22:59) compared with departure passengers in the same hours on other days, and what does that cost us over the six months in total?")

**Expected:** the agent compares spend per passenger, scales up the 3% sample (about x33) and gives a revenue-loss estimate of roughly AED 600,000 to 700,000 over the six months (spend per passenger drops from about 51 to about 36 AED on Friday evenings). Ask the room: "Would you have caught this in a monthly dashboard?"

### Beat 4: What are the rules?

In [ ]:
ask("What do our service standards say about the maximum departure immigration wait and the minimum number of counters at peak hours?")

**Expected:** a cited answer from the Passenger Service Standards (sections 2.3 and 3.2). This is RAG: the answer comes from our document, with the section number so it can be checked.

### Beat 5: The "aha" moment. Facts against rules, and what to do

In [ ]:
ask("Compare what actually happens on Friday evenings against our service standards: waits, counters open and missed-flight rate. Which standards are we breaking, what is the financial cost, and what should we do next? Use the complaint procedure to say what the process requires from here.")

**Expected:** one answer that uses both tools: breaches of PSS-01 sections 2.3, 3.2 and 5.1, the revenue estimate, and the next steps from the complaint procedure (a systemic issue needing a root cause report within 14 days).

**Pause here.** This is the moment to tell the room: one database, four documents, two tools and a good prompt. That is the whole idea of an agent.

### Bonus beats (use for Q&A or the extended session)

**Bonus A: What are passengers actually saying?** (English and Arabic comments)

In [ ]:
ask("What do departing passengers complain about on Friday evenings? Summarise the main themes from the low-score comments, including the Arabic ones, and tell me which category scores worst compared with other times.")

**Bonus B: How do we get more people to travel through the airport?**

In [ ]:
ask("Which routes qualify as priority growth routes under our airline incentive policy, using the last three months of data? What incentive could we offer and what condition must the airline meet?")

**Expected:** one route stands out with an extremely high load factor and very few weekly departures. The answer must quote the incentive policy conditions.

**Bonus C: Which airline is hurting us?**

In [ ]:
ask("Which airline is breaching our delay policy, how bad is it compared with the limit, and what are the consequences under the policies?")

**Expected:** one airline has a delayed-flight rate roughly double the 30 percent limit, which triggers a formal warning and affects incentives (Delay Management SOP section 2 and Airline Incentive Policy section 4).

## Presenter answer key (run before the session and keep it open)

This cell computes the true answers directly from the database, so you can check what the agent says. Numbers are the same for everyone because of the fixed seed, but they may differ slightly if the library versions differ.

In [ ]:
con = sqlite3.connect(DB_PATH)

def show(title, sql):
    print("\n" + title)
    display(pd.read_sql(sql, con))

show("1. Average departure immigration wait by weekday", """
SELECT weekday, ROUND(AVG(immigration_wait_min),1) AS avg_wait
FROM passenger_journeys WHERE direction='Departure' GROUP BY weekday ORDER BY avg_wait DESC""")

show("2. Friday 17:00-22:59 vs all other departure journeys", """
SELECT CASE WHEN weekday='Friday' AND hour BETWEEN 17 AND 22 THEN 'Friday evening' ELSE 'All other' END AS slot,
       COUNT(*) AS sampled_journeys,
       ROUND(AVG(immigration_wait_min),1) AS avg_wait,
       ROUND(AVG(spend_aed),1) AS avg_spend_aed,
       ROUND(100.0*AVG(missed_flight),2) AS missed_flight_pct
FROM passenger_journeys WHERE direction='Departure' GROUP BY slot""")

show("3. Counters open at peak hours (17-22): Friday vs other days", """
SELECT CASE WHEN strftime('%w', flight_date)='5' THEN 'Friday' ELSE 'Other days' END AS day_group,
       ROUND(AVG(counters_open),1) AS avg_counters
FROM staffing WHERE direction='Departure' AND hour BETWEEN 17 AND 22 GROUP BY day_group""")

show("4. Estimated spend lost on Friday evenings over six months (AED, 3% sample scaled up)", """
WITH s AS (
  SELECT AVG(CASE WHEN NOT (weekday='Friday' AND hour BETWEEN 17 AND 22) AND hour BETWEEN 17 AND 22 THEN spend_aed END) AS other_eve,
         AVG(CASE WHEN weekday='Friday' AND hour BETWEEN 17 AND 22 THEN spend_aed END) AS fri_eve,
         SUM(CASE WHEN weekday='Friday' AND hour BETWEEN 17 AND 22 THEN 1 ELSE 0 END) AS n_fri
  FROM passenger_journeys WHERE direction='Departure')
SELECT ROUND(other_eve,1) AS other_evenings_avg_spend, ROUND(fri_eve,1) AS friday_evening_avg_spend,
       ROUND((other_eve-fri_eve)*n_fri/0.03) AS estimated_loss_aed FROM s""")

show("5. Airline delay rate (limit in policy: 30%)", """
SELECT airline, ROUND(100.0*SUM(status='Delayed')/COUNT(*),1) AS delayed_pct
FROM flights WHERE status!='Cancelled' GROUP BY airline ORDER BY delayed_pct DESC""")

show("6. Priority growth routes: departures, last 3 months (load factor >= 0.90 and < 7 weekly departures)", """
SELECT route, ROUND(AVG(load_factor),3) AS avg_load_factor, ROUND(COUNT(*)/13.0,1) AS weekly_departures
FROM flights WHERE direction='Departure' AND status!='Cancelled' AND flight_date >= '2026-07-01'
GROUP BY route ORDER BY avg_load_factor DESC LIMIT 5""")

con.close()

## Step 8. Give it a face: a chat interface

**Say:** "A notebook is for builders. Managers need a chat window. Same agent, same tools, now with a screen."

The interface also prints which tools were used under every answer, so users can trust but verify.

In [22]:
import gradio as gr

ui_chat = client.chats.create(model=MODEL, config=config)


def respond(message, history):
    TOOL_LOG.clear()
    reply = ui_chat.send_message(message).text
    used = "\n".join(f"- `{name}`" for name, _, _ in TOOL_LOG)
    return reply + (f"\n\n---\n*Tools used:*\n{used}" if used else "")


demo = gr.ChatInterface(
    fn=respond,
    title="Airport Operations and Growth Analyst",
    description="Ask about operations, passenger experience, revenue and growth. All data is fictional.",
    examples=[
        "Which weekday has the longest departure immigration wait?",
        "What does our service standard say about counters at peak hours?",
        "Which airline is breaching the delay policy?",
        "Which routes should we add flights to?",
    ],
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba8a5a60d24a7bc037.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## What you just built

- A synthetic **database** and a set of **documents** for a fictional airport
- A **SQL tool** with guardrails and a **RAG tool** with citations
- A **system prompt** that sets the rules
- An **agent loop** that decides which tool to use, and a **chat interface**

**Swap the domain, keep the pattern.** Replace the database with HR, energy, finance or service requests, replace the documents with that domain's policies, and the same notebook becomes a new agent.

## Before you take this to production

- Use a real read-only database account, not only a keyword filter
- Add query time limits and row limits at the database level
- Log every tool call and keep the logs
- Build a test list of questions with known answers and run it every time the prompt, model or data changes
- Decide what data may leave your organisation before it reaches any external API

## Keep going

This is the pattern behind the AI Residency and the AI Accelerator Bootcamp: you build agents like this one, with feedback, templates and a community around you.